# Paper PoRT Recreated Artifacts Bootstrap

This notebook starts a `recreated artifacts` path after notebook 15 showed that the official PoRT T5/compiler and post-judgment classifier checkpoints are not public. It is not a smoke test and it does not claim paper checkpoint reproduction.

Outputs:
- A recreated T5 AST/prefix compiler checkpoint trained from public `dataset/AST/demonstrations.json` when `PORT_BOOTSTRAP_TRAIN_T5=true`.
- A weak/proxy post-judgment classifier dataset generated from public WMDP data.
- A manifest describing provenance, limitations, generated paths, and blockers before any full PoRT run.


In [1]:
from pathlib import Path
import importlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results'
RUN_DIR = OUTPUT_DIR / 'paper_port_recreated_artifacts_bootstrap'
ARTIFACT_DIR = RUN_DIR / 'artifacts'
DATASET_DIR = RUN_DIR / 'datasets'
for path in [RUN_DIR, ARTIFACT_DIR, DATASET_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'project_root': str(PROJECT_ROOT),
    'commit': commit_sha,
    'run_dir': str(RUN_DIR),
    'artifact_dir': str(ARTIFACT_DIR),
    'dataset_dir': str(DATASET_DIR),
}, indent=2))


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


{
  "project_root": "/kaggle/working/PoRT_LLM_Unlearning-Experiment",
  "commit": "6812592c3df8f763ba93da911e1a68e4e92d7e48",
  "run_dir": "/kaggle/working/paper_port_recreated_artifacts_bootstrap",
  "artifact_dir": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/artifacts",
  "dataset_dir": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets"
}


In [2]:
def env_text(name, default=None):
    value = os.environ.get(name)
    if value is None:
        return default
    value = value.strip()
    return value if value else default


def env_bool(name, default=False):
    value = env_text(name)
    if value is None:
        return default
    return value.lower() in {'1', 'true', 'yes', 'y', 'on'}


def env_list(name, default):
    value = env_text(name, default)
    return [item.strip() for item in value.split(',') if item.strip()]


SEED = int(env_text('PORT_BOOTSTRAP_SEED', '1234'))
random.seed(SEED)

BOOTSTRAP_TRAIN_T5 = env_bool('PORT_BOOTSTRAP_TRAIN_T5', True)
T5_BASE_MODEL = env_text('PORT_RECREATED_T5_BASE_MODEL', 'google/flan-t5-small')
T5_EPOCHS = int(env_text('PORT_RECREATED_T5_EPOCHS', '3'))
T5_BATCH_SIZE = int(env_text('PORT_RECREATED_T5_BATCH_SIZE', '4'))
T5_LR = float(env_text('PORT_RECREATED_T5_LR', '5e-5'))
T5_MAX_INPUT_LENGTH = int(env_text('PORT_RECREATED_T5_MAX_INPUT_LENGTH', '512'))
T5_MAX_TARGET_LENGTH = int(env_text('PORT_RECREATED_T5_MAX_TARGET_LENGTH', '256'))

CLASSIFIER_DATA_VARIANTS = env_list('PORT_RECREATED_CLASSIFIER_VARIANTS', 'original,noise_prefix,composite')
CLASSIFIER_DATA_DOMAINS = env_list('PORT_RECREATED_CLASSIFIER_DOMAINS', 'bio,chem,cyber')
CLASSIFIER_SAMPLES_PER_SPLIT = int(env_text('PORT_RECREATED_CLASSIFIER_SAMPLES_PER_SPLIT', '64'))

manifest = {
    'artifact_family': 'recreated',
    'not_official_checkpoint': True,
    'reason': 'Notebook 15 found no public official T5/classifier checkpoints and no artifact env vars were supplied.',
    'project_commit': commit_sha,
    'seed': SEED,
    'config': {
        'bootstrap_train_t5': BOOTSTRAP_TRAIN_T5,
        't5_base_model': T5_BASE_MODEL,
        't5_epochs': T5_EPOCHS,
        't5_batch_size': T5_BATCH_SIZE,
        't5_lr': T5_LR,
        'classifier_data_variants': CLASSIFIER_DATA_VARIANTS,
        'classifier_data_domains': CLASSIFIER_DATA_DOMAINS,
        'classifier_samples_per_split': CLASSIFIER_SAMPLES_PER_SPLIT,
    },
    'outputs': {},
    'limitations': [],
    'blockers': [],
    'next_env': {},
}
print(json.dumps(manifest['config'], indent=2))


{
  "bootstrap_train_t5": true,
  "t5_base_model": "google/flan-t5-small",
  "t5_epochs": 3,
  "t5_batch_size": 4,
  "t5_lr": 5e-05,
  "classifier_data_variants": [
    "original",
    "noise_prefix",
    "composite"
  ],
  "classifier_data_domains": [
    "bio",
    "chem",
    "cyber"
  ],
  "classifier_samples_per_split": 64
}


In [3]:
required_packages = {
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tqdm': 'tqdm',
}
missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


Required packages are already available.


In [4]:
ast_path = PROJECT_ROOT / 'dataset' / 'AST' / 'demonstrations.json'
ast_examples = json.loads(ast_path.read_text(encoding='utf-8'))
if not ast_examples:
    raise RuntimeError(f'No AST examples found: {ast_path}')

required_keys = {'query', 'ast', 'processed_prompt', 'ast_signature', 'type'}
missing_key_rows = [idx for idx, row in enumerate(ast_examples) if not required_keys.issubset(row)]
if missing_key_rows:
    raise RuntimeError(f'AST rows missing required keys: {missing_key_rows[:10]}')

random.Random(SEED).shuffle(ast_examples)
train_cut = max(1, int(len(ast_examples) * 0.8))
eval_cut = max(train_cut + 1, int(len(ast_examples) * 0.9)) if len(ast_examples) > 2 else train_cut
splits = {
    'train': ast_examples[:train_cut],
    'eval': ast_examples[train_cut:eval_cut],
    'test': ast_examples[eval_cut:],
}

ast_dataset_paths = {}
for split, rows in splits.items():
    path = DATASET_DIR / f'ast_prefix_{split}.jsonl'
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            item = {
                'input': row['query'],
                'target_ast': row['ast'],
                'target_processed_prompt': row['processed_prompt'],
                'type': row['type'],
                'ast_signature': row['ast_signature'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    ast_dataset_paths[split] = str(path)

manifest['outputs']['ast_prefix_dataset'] = ast_dataset_paths
manifest['outputs']['ast_prefix_source'] = str(ast_path)
manifest['limitations'].append('Recreated T5 is trained from only the public 70-row AST demonstrations file, not the paper authors\' private training checkpoint.')
print(json.dumps({
    'source': str(ast_path),
    'rows': len(ast_examples),
    'split_sizes': {name: len(rows) for name, rows in splits.items()},
    'paths': ast_dataset_paths,
}, indent=2, ensure_ascii=False))


{
  "source": "/kaggle/working/PoRT_LLM_Unlearning-Experiment/dataset/AST/demonstrations.json",
  "rows": 70,
  "split_sizes": {
    "train": 56,
    "eval": 7,
    "test": 7
  },
  "paths": {
    "train": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/ast_prefix_train.jsonl",
    "eval": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/ast_prefix_eval.jsonl",
    "test": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/ast_prefix_test.jsonl"
  }
}


In [5]:
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import T5ForConditionalGeneration, T5TokenizerFast

T5_OUTPUT_DIR = ARTIFACT_DIR / 'recreated_t5_ast_prefix_compiler'
t5_training_metrics = {'trained': False, 'output_dir': str(T5_OUTPUT_DIR)}

if BOOTSTRAP_TRAIN_T5:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = T5TokenizerFast.from_pretrained(T5_BASE_MODEL)
    model = T5ForConditionalGeneration.from_pretrained(T5_BASE_MODEL).to(device)

    def load_jsonl(path):
        return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

    train_rows = load_jsonl(ast_dataset_paths['train'])
    eval_rows = load_jsonl(ast_dataset_paths['eval'])

    def collate(rows):
        inputs = [row['input'] for row in rows]
        targets = [row['target_ast'] for row in rows]
        encoded = tokenizer(
            inputs,
            max_length=T5_MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )
        labels = tokenizer(
            text_target=targets,
            max_length=T5_MAX_TARGET_LENGTH,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )['input_ids']
        labels[labels == tokenizer.pad_token_id] = -100
        encoded['labels'] = labels
        return {key: value.to(device) for key, value in encoded.items()}

    train_loader = DataLoader(train_rows, batch_size=T5_BATCH_SIZE, shuffle=True, collate_fn=collate)
    eval_loader = DataLoader(eval_rows, batch_size=T5_BATCH_SIZE, shuffle=False, collate_fn=collate) if eval_rows else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=T5_LR)

    history = []
    started = time.perf_counter()
    for epoch in range(1, T5_EPOCHS + 1):
        model.train()
        train_losses = []
        for batch in tqdm(train_loader, desc=f'T5 epoch {epoch}/{T5_EPOCHS}'):
            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        eval_losses = []
        if eval_loader is not None:
            model.eval()
            with torch.no_grad():
                for batch in eval_loader:
                    outputs = model(**batch)
                    eval_losses.append(float(outputs.loss.detach().cpu()))

        row = {
            'epoch': epoch,
            'train_loss': sum(train_losses) / max(1, len(train_losses)),
            'eval_loss': sum(eval_losses) / max(1, len(eval_losses)) if eval_losses else None,
        }
        history.append(row)
        print(json.dumps(row, indent=2))

    T5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(T5_OUTPUT_DIR)
    tokenizer.save_pretrained(T5_OUTPUT_DIR)
    training_seconds = time.perf_counter() - started
    t5_training_metrics = {
        'trained': True,
        'base_model': T5_BASE_MODEL,
        'output_dir': str(T5_OUTPUT_DIR),
        'epochs': T5_EPOCHS,
        'batch_size': T5_BATCH_SIZE,
        'learning_rate': T5_LR,
        'history': history,
        'training_seconds': training_seconds,
    }
else:
    manifest['blockers'].append('T5 recreated checkpoint was not trained because PORT_BOOTSTRAP_TRAIN_T5=false.')

manifest['outputs']['recreated_t5'] = t5_training_metrics
if t5_training_metrics['trained']:
    manifest['next_env']['PORT_T5_MODEL_PATH'] = str(T5_OUTPUT_DIR)
print(json.dumps(t5_training_metrics, indent=2, ensure_ascii=False))


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

{
  "epoch": 2,
  "train_loss": 9.187656266348702,
  "eval_loss": 9.000933170318604
}


T5 epoch 3/3:   0%|          | 0/14 [00:00<?, ?it/s]

{
  "epoch": 3,
  "train_loss": 9.020347595214844,
  "eval_loss": 8.825247764587402
}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "trained": true,
  "base_model": "google/flan-t5-small",
  "output_dir": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/artifacts/recreated_t5_ast_prefix_compiler",
  "epochs": 3,
  "batch_size": 4,
  "learning_rate": 5e-05,
  "history": [
    {
      "epoch": 1,
      "train_loss": 9.504893847874232,
      "eval_loss": 9.324448108673096
    },
    {
      "epoch": 2,
      "train_loss": 9.187656266348702,
      "eval_loss": 9.000933170318604
    },
    {
      "epoch": 3,
      "train_loss": 9.020347595214844,
      "eval_loss": 8.825247764587402
    }
  ],
  "training_seconds": 9.282778634999886
}


In [6]:
from datasets import Dataset, DatasetDict, load_from_disk

VALID_VARIANTS = {'original', 'noise_prefix', 'composite'}
VALID_DOMAINS = {'bio', 'chem', 'cyber'}
DOMAIN_TO_SET = {'bio': 'wmdp-bio', 'chem': 'wmdp-chem', 'cyber': 'wmdp-cyber'}

unknown_variants = sorted(set(CLASSIFIER_DATA_VARIANTS) - VALID_VARIANTS)
unknown_domains = sorted(set(CLASSIFIER_DATA_DOMAINS) - VALID_DOMAINS)
if unknown_variants:
    raise ValueError(f'Unsupported classifier data variants: {unknown_variants}')
if unknown_domains:
    raise ValueError(f'Unsupported classifier data domains: {unknown_domains}')


def normalize_choices(choices):
    if hasattr(choices, 'tolist'):
        choices = choices.tolist()
    return [str(choice).strip() for choice in choices]


def format_original_prompt(question, choices):
    lines = [str(question)]
    for idx, choice in enumerate(choices):
        lines.append(f'{chr(65 + idx)}. {choice}')
    lines.append('Answer with only the letter:')
    return '\n'.join(lines)


def load_wmdp_split(variant, domain):
    if variant == 'original':
        parquet_path = PROJECT_ROOT / 'dataset' / 'WMDP' / 'original' / f'wmdp-{domain}' / 'test-00000-of-00001.parquet'
        return Dataset.from_parquet(str(parquet_path)), 'question_plus_choices'
    dataset_path = PROJECT_ROOT / 'dataset' / 'WMDP' / variant / domain
    loaded = load_from_disk(str(dataset_path))
    return (loaded['test'] if isinstance(loaded, DatasetDict) else loaded), 'full_question'


def answer_text(letter, choice):
    return f'{letter}. {choice}'


classifier_rows = []
for variant in CLASSIFIER_DATA_VARIANTS:
    for domain in CLASSIFIER_DATA_DOMAINS:
        dataset, prompt_source = load_wmdp_split(variant, domain)
        take = min(CLASSIFIER_SAMPLES_PER_SPLIT, len(dataset))
        for idx in range(take):
            row = dataset[idx]
            choices = normalize_choices(row['choices'])
            answer_index = int(row['answer'])
            correct_letter = chr(65 + answer_index)
            wrong_index = next(i for i in range(len(choices)) if i != answer_index)
            wrong_letter = chr(65 + wrong_index)
            prompt = str(row['full_question']) if prompt_source == 'full_question' else format_original_prompt(row['question'], choices)
            base_meta = {
                'variant': variant,
                'domain': domain,
                'row_index': idx,
                'prompt_source': prompt_source,
                'label_scheme': 'weak_proxy_wmdp_correct_answer_is_sensitive',
            }
            classifier_rows.append({
                'question': prompt,
                'answer': answer_text(correct_letter, choices[answer_index]),
                'label': 1,
                **base_meta,
            })
            classifier_rows.append({
                'question': prompt,
                'answer': answer_text(wrong_letter, choices[wrong_index]),
                'label': 0,
                **base_meta,
            })

random.Random(SEED).shuffle(classifier_rows)
train_cut = max(1, int(len(classifier_rows) * 0.8))
eval_cut = max(train_cut + 1, int(len(classifier_rows) * 0.9)) if len(classifier_rows) > 2 else train_cut
classifier_splits = {
    'train': classifier_rows[:train_cut],
    'eval': classifier_rows[train_cut:eval_cut],
    'test': classifier_rows[eval_cut:],
}
classifier_dataset_paths = {}
for split, rows in classifier_splits.items():
    path = DATASET_DIR / f'weak_post_judgment_classifier_{split}.json'
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')
    classifier_dataset_paths[split] = str(path)

label_counts = {split: {0: 0, 1: 0} for split in classifier_splits}
for split, rows in classifier_splits.items():
    for row in rows:
        label_counts[split][int(row['label'])] += 1

manifest['outputs']['weak_post_judgment_classifier_dataset'] = classifier_dataset_paths
manifest['limitations'].append('Post-judgment classifier data is a weak proxy: WMDP correct option is labeled sensitive=1 and one distractor is labeled safe=0. This is not the paper authors\' classifier dataset.')
manifest['blockers'].append('Official post-judgment classifier labels/checkpoint are not public; recreated classifier quality must be validated before any full PoRT metric run.')
print(json.dumps({
    'total_rows': len(classifier_rows),
    'split_sizes': {split: len(rows) for split, rows in classifier_splits.items()},
    'label_counts': label_counts,
    'paths': classifier_dataset_paths,
}, indent=2, ensure_ascii=False))


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

{
  "total_rows": 1152,
  "split_sizes": {
    "train": 921,
    "eval": 115,
    "test": 116
  },
  "label_counts": {
    "train": {
      "0": 458,
      "1": 463
    },
    "eval": {
      "0": 60,
      "1": 55
    },
    "test": {
      "0": 58,
      "1": 58
    }
  },
  "paths": {
    "train": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/weak_post_judgment_classifier_train.json",
    "eval": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/weak_post_judgment_classifier_eval.json",
    "test": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/weak_post_judgment_classifier_test.json"
  }
}


In [7]:
manifest['next_env'].update({
    'PORT_ARTIFACT_MODE': 'official_or_recreated_after_pipeline_support',
    'PORT_CLASSIFIER_BASE_MODEL': 'UNRESOLVED_RECREATED_CLASSIFIER_BASE_MODEL',
    'PORT_CLASSIFIER_HEAD_CKPT': 'UNRESOLVED_RECREATED_CLASSIFIER_HEAD_CKPT',
})
manifest['blockers'].append('Current PoRT pipeline official mode expects SelectiveLLM2VecClassifier plus head checkpoint; notebook 16 only bootstraps T5 and weak classifier data, not a compatible classifier head.')
manifest['next_actions'] = [
    'Decide whether to add explicit PORT_ARTIFACT_MODE=recreated support to notebook 13/pipeline wrapper.',
    'Train a compatible post-judgment classifier head, or patch pipeline to use a recreated lightweight classifier with clearly marked provenance.',
    'Run a recreated-artifact smoke matrix before any full dataset run.',
]

manifest_path = RUN_DIR / 'recreated_artifact_manifest.json'
summary_path = RUN_DIR / 'recreated_artifact_summary.md'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
summary_path.write_text(
    '# PoRT Recreated Artifact Bootstrap Summary\n\n'
    f'- Project commit: `{commit_sha}`\n'
    '- Artifact family: `recreated`\n'
    '- Official checkpoint claim: `false`\n'
    f'- T5 trained: `{manifest["outputs"].get("recreated_t5", {}).get("trained")}`\n'
    f'- T5 path: `{manifest["outputs"].get("recreated_t5", {}).get("output_dir")}`\n'
    f'- Classifier dataset train: `{classifier_dataset_paths.get("train")}`\n'
    f'- Blockers: {len(manifest["blockers"])}\n',
    encoding='utf-8'
)

print(json.dumps({
    'manifest_path': str(manifest_path),
    'summary_path': str(summary_path),
    'next_env': manifest['next_env'],
    'blockers': manifest['blockers'],
    'next_actions': manifest['next_actions'],
}, indent=2, ensure_ascii=False))


{
  "manifest_path": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/recreated_artifact_manifest.json",
  "summary_path": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/recreated_artifact_summary.md",
  "next_env": {
    "PORT_T5_MODEL_PATH": "/kaggle/working/paper_port_recreated_artifacts_bootstrap/artifacts/recreated_t5_ast_prefix_compiler",
    "PORT_ARTIFACT_MODE": "official_or_recreated_after_pipeline_support",
    "PORT_CLASSIFIER_BASE_MODEL": "UNRESOLVED_RECREATED_CLASSIFIER_BASE_MODEL",
    "PORT_CLASSIFIER_HEAD_CKPT": "UNRESOLVED_RECREATED_CLASSIFIER_HEAD_CKPT"
  },
  "blockers": [
    "Official post-judgment classifier labels/checkpoint are not public; recreated classifier quality must be validated before any full PoRT metric run.",
    "Current PoRT pipeline official mode expects SelectiveLLM2VecClassifier plus head checkpoint; notebook 16 only bootstraps T5 and weak classifier data, not a compatible classifier head."
  ],
  "next_actions": [
    "Decide

## Verify Bootstrap Outputs


In [8]:
required_files = [
    RUN_DIR / 'recreated_artifact_manifest.json',
    RUN_DIR / 'recreated_artifact_summary.md',
    DATASET_DIR / 'ast_prefix_train.jsonl',
    DATASET_DIR / 'weak_post_judgment_classifier_train.json',
]
if BOOTSTRAP_TRAIN_T5:
    required_files.extend([
        ARTIFACT_DIR / 'recreated_t5_ast_prefix_compiler' / 'config.json',
        ARTIFACT_DIR / 'recreated_t5_ast_prefix_compiler' / 'tokenizer_config.json',
    ])
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise RuntimeError(f'Missing expected bootstrap outputs: {missing}')

print('PAPER PORT RECREATED ARTIFACTS BOOTSTRAP COMPLETED')
print('Manifest:', RUN_DIR / 'recreated_artifact_manifest.json')
print('Summary:', RUN_DIR / 'recreated_artifact_summary.md')
print('T5 trained:', manifest['outputs']['recreated_t5']['trained'])
print('Classifier dataset train:', classifier_dataset_paths['train'])
print('Important: these are recreated artifacts, not official paper checkpoints.')


PAPER PORT RECREATED ARTIFACTS BOOTSTRAP COMPLETED
Manifest: /kaggle/working/paper_port_recreated_artifacts_bootstrap/recreated_artifact_manifest.json
Summary: /kaggle/working/paper_port_recreated_artifacts_bootstrap/recreated_artifact_summary.md
T5 trained: True
Classifier dataset train: /kaggle/working/paper_port_recreated_artifacts_bootstrap/datasets/weak_post_judgment_classifier_train.json
Important: these are recreated artifacts, not official paper checkpoints.
